# PEMDT-Cirrhosis — Early-Warning of Hepatic Decompensation

**Prospective landmark design · time-censored · MIMIC-IV cirrhosis cohort**

Goal: given only what is known in the **first `LANDMARK_HOURS` of admission**, predict
whether a cirrhosis patient will go on to **decompensate later in the stay**. This is a
genuine early-warning task, not concurrent detection.

## Why this design (read before interpreting)
- **Landmark + horizon.** Features use only windows in `[admit, admit+L]`. The label is
  whether the first *timestamped* decompensation marker occurs **after** L. This removes
  the concurrency leakage in the earlier notebook (where MELD-Na and the label shared the
  same admission window).
- **Incident, not prevalent.** Patients whose decompensation marker occurs **at/before L**
  are *already decompensated* — excluded (they are not an early-warning opportunity). In
  this cohort most markers are prevalent (median ~5 h after admit), so the incident group
  is the minority and is the scientifically correct denominator.
- **Clean labels.** Controls = no decompensation marker **and** ICD-negative for
  decompensation. Patients who are ICD-positive but have no timestamped marker are
  **ambiguous** (decompensated but un-timed) and are *excluded*, not dumped into controls.
- **Robust scaling.** Population z-scores use median + IQR (not mean/SD), because liver
  labs are heavily right-tailed (AST σ≈640 in this cohort). Fixes the signal-dilution seen
  before.
- **Validated lab baselines.** FIB-4, APRI, AST/ALT ratio and MELD-Na are computed from
  pre-landmark labs and used **both** as model features **and** as standalone baselines —
  so we can see whether the HMM trajectory adds anything beyond a static lab score.
- **Honest evaluation.** Patient-level stratified train/test split; all metrics reported on
  the held-out **test set** with bootstrap CIs. No training-set AUC.

## Known limitations (carried forward honestly)
- "Decompensation onset" is anchored by drug/procedure markers (lactulose, rifaximin,
  terlipressin, octreotide, paracentesis); some true events are missed or mistimed.
- `GCS_total` and `HRV_proxy` from the source timeline are excluded (the former was
  mis-constructed from GCS *components*; the latter is mostly an imputed constant).
- This is a retrospective ICU cohort: it cannot represent truly *early/asymptomatic*
  disease, only in-hospital incident decompensation. Presented for review, not clinical use.

**Inputs:** `cirrhosis_cohort.csv`, `cirrhosis_timeline.csv` (produced by `PEMDT_cirrhosis.ipynb`).


## Config

In [1]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

DATA_DIR = Path("data")                       # folder holding the two input CSVs
COHORT_CSV   = DATA_DIR / "cirrhosis_cohort.csv"
TIMELINE_CSV = DATA_DIR / "cirrhosis_timeline.csv"
OUT = Path("results/early_warning_output"); OUT.mkdir(exist_ok=True)

# ---- Landmark design parameters ----------------------------
LANDMARK_HOURS = 24      # observation window length from admission
HORIZON_HOURS  = None    # None = predict decompensation any time after L during the stay;
                         # or set e.g. 168 for "within 7 days after the landmark"
MIN_OBS_HOURS  = 6       # require at least this much pre-landmark data
AST_ULN        = 40.0    # upper-limit-of-normal for APRI
RANDOM_SEED    = 42
TEST_FRAC      = 0.30
CHUNK          = 200_000

# ---- HMM features (GCS_total & HRV_proxy excluded on purpose) ----
HMM_FEATURES = ["Bilirubin","Albumin","INR","Platelets","Creatinine","Sodium",
                "Ammonia","WBC","AST","ALT","Lactate","Hemoglobin",
                "MAP","HR_bpm","SpO2","Resp_rate"]
DECOMP_Z = {  # direction of decompensation per feature
    "Bilirubin":+2.0,"Albumin":-2.0,"INR":+2.0,"Platelets":-2.0,"Creatinine":+2.0,
    "Sodium":-2.0,"Ammonia":+2.0,"WBC":+1.5,"AST":+1.5,"ALT":+1.0,"Lactate":+2.0,
    "Hemoglobin":-1.5,"MAP":-2.0,"HR_bpm":+1.5,"SpO2":-1.5,"Resp_rate":+1.5,
}
FEATURE_WEIGHTS = {
    "Bilirubin":1.6,"Albumin":1.4,"INR":1.5,"Platelets":1.3,"Creatinine":1.4,
    "Sodium":1.3,"Ammonia":1.5,"WBC":1.0,"AST":0.8,"ALT":0.6,"Lactate":1.4,
    "Hemoglobin":1.0,"MAP":1.3,"HR_bpm":1.0,"SpO2":1.0,"Resp_rate":1.0,
}
print("Config loaded. Landmark =", LANDMARK_HOURS, "h | horizon =", HORIZON_HOURS)


Config loaded. Landmark = 24 h | horizon = None


## Step 1 — Landmark cohort

In [2]:
# ============================================================
# STEP 1 — LANDMARK COHORT (cases / controls / exclusions)
# ============================================================
def build_landmark_cohort():
    co = pd.read_csv(COHORT_CSV)
    co["admittime"] = pd.to_datetime(co["admittime"], errors="coerce")
    co["dischtime"] = pd.to_datetime(co["dischtime"], errors="coerce")
    co["first_decomp_marker_time"] = pd.to_datetime(co["first_decomp_marker_time"], errors="coerce")

    L = LANDMARK_HOURS
    co["hrs_to_marker"] = (co["first_decomp_marker_time"] - co["admittime"]).dt.total_seconds()/3600
    co["los_hours"]     = (co["dischtime"] - co["admittime"]).dt.total_seconds()/3600
    co["los_hours"]     = co["los_hours"].fillna(co["los_days"]*24)

    has_marker = co["first_decomp_marker_time"].notna()
    icd_decomp = co["decompensated"] == 1

    prevalent  = has_marker & (co["hrs_to_marker"] <= L)         # already decompensated by L
    incident   = has_marker & (co["hrs_to_marker"] >  L)         # decompensates after L
    if HORIZON_HOURS is not None:
        incident &= co["hrs_to_marker"] <= (L + HORIZON_HOURS)
    ambiguous  = icd_decomp & (~has_marker)                      # decompensated but un-timed
    clean_ctrl = (~has_marker) & (~icd_decomp)                   # never decompensated

    at_risk = co["los_hours"] > L                                # observed beyond the landmark

    co["ew_label"] = np.nan
    co.loc[incident & at_risk, "ew_label"]   = 1
    co.loc[clean_ctrl & at_risk, "ew_label"] = 0

    keep = co["ew_label"].notna() & (~prevalent) & (~ambiguous)
    cohort = co[keep].copy()
    cohort["ew_label"] = cohort["ew_label"].astype(int)

    print("CONSORT FLOW")
    print(f"  total cirrhosis patients .............. {len(co):,}")
    print(f"  prevalent (marker <= {L}h, excluded) .. {int((prevalent).sum()):,}")
    print(f"  ambiguous (ICD+ but un-timed, excl) ... {int((ambiguous & ~prevalent).sum()):,}")
    print(f"  not at-risk past landmark (excluded) .. {int((~at_risk).sum()):,}")
    print(f"  ---------------------------------------")
    print(f"  ANALYSIS COHORT ....................... {len(cohort):,}")
    print(f"    incident cases (label=1) ............ {int((cohort['ew_label']==1).sum()):,}")
    print(f"    clean controls (label=0) ............ {int((cohort['ew_label']==0).sum()):,}")
    print(f"    event rate .......................... {cohort['ew_label'].mean()*100:.1f}%")
    return cohort

cohort = build_landmark_cohort()


CONSORT FLOW
  total cirrhosis patients .............. 7,958
  prevalent (marker <= 24h, excluded) .. 3,572
  ambiguous (ICD+ but un-timed, excl) ... 484
  not at-risk past landmark (excluded) .. 665
  ---------------------------------------
  ANALYSIS COHORT ....................... 3,455
    incident cases (label=1) ............ 1,017
    clean controls (label=0) ............ 2,438
    event rate .......................... 29.4%


## Step 2 — Robust population stats + time-censored HMM

In [3]:
# ============================================================
# STEP 2 — ROBUST POPULATION STATS + TIME-CENSORED HMM
# ============================================================
from scipy.stats import norm

def robust_stats(timeline_csv):
    """median + IQR-based SD per feature, streamed over the whole timeline."""
    acc = {f: [] for f in HMM_FEATURES}
    for ch in pd.read_csv(timeline_csv, chunksize=CHUNK,
                          usecols=[c for c in HMM_FEATURES], low_memory=False):
        for f in HMM_FEATURES:
            v = ch[f].dropna().values
            if len(v): acc[f].append(v)
    stats = {}
    for f in HMM_FEATURES:
        if acc[f]:
            v = np.concatenate(acc[f])
            if len(v) > 100:
                med = float(np.median(v))
                iqr = float(np.percentile(v, 75) - np.percentile(v, 25))
                sd  = iqr / 1.349 if iqr > 1e-6 else float(np.std(v))
                if not np.isfinite(sd) or sd < 1e-6: sd = 1.0
                stats[f] = (med, sd)
    print(f"✅ Robust stats for {len(stats)} features")
    return stats

def hmm_trajectory(df, stats):
    """Looser-dynamics Bayesian HMM on one patient's pre-landmark windows."""
    df = df.sort_values("timestamp").reset_index(drop=True)
    belief = np.array([0.80, 0.20])
    base_T = np.array([[0.88, 0.12], [0.18, 0.82]])   # looser than the v1 notebook
    probs = []
    for _, row in df.iterrows():
        T = base_T.copy()
        if belief[1] > 0.5:
            T[1, 1], T[1, 0] = 0.92, 0.08
        pred = belief @ T
        lls = llu = 0.0
        for f in HMM_FEATURES:
            if f not in stats: continue
            val = row.get(f, np.nan)
            if pd.isna(val): continue
            med, sd = stats[f]
            z = np.clip((val - med) / sd, -8, 8)       # clip extreme robust-z
            w, zu = FEATURE_WEIGHTS[f], DECOMP_Z[f]
            lls += w * norm.logpdf(z, 0.0, 1.0)
            llu += w * norm.logpdf(z, zu, 1.5)
        lps, lpu = np.log(pred[0]+1e-300)+lls, np.log(pred[1]+1e-300)+llu
        m = max(lps, lpu)
        ps, pu = np.exp(lps-m), np.exp(lpu-m)
        belief = np.array([ps, pu]) / (ps+pu+1e-9)
        probs.append(belief[1])
    return np.array(probs), df["hrs"].values

def run_prelandmark_hmm(stats, cohort):
    """Stream timeline, keep pre-landmark windows for cohort patients, run HMM, aggregate."""
    admit = dict(zip(cohort["patient_id"], pd.to_datetime(cohort["admittime"])))
    pid_set = set(cohort["patient_id"])
    parts = []
    cols = ["patient_id","timestamp"] + HMM_FEATURES
    for ch in pd.read_csv(TIMELINE_CSV, chunksize=CHUNK, usecols=cols, low_memory=False):
        ch = ch[ch["patient_id"].isin(pid_set)]
        if ch.empty: continue
        ch["timestamp"] = pd.to_datetime(ch["timestamp"], errors="coerce")
        ch["adm"] = ch["patient_id"].map(admit)
        ch["hrs"] = (ch["timestamp"] - ch["adm"]).dt.total_seconds()/3600
        ch = ch[(ch["hrs"] >= 0) & (ch["hrs"] <= LANDMARK_HOURS)]
        if not ch.empty: parts.append(ch)
    pre = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    print(f"  pre-landmark windows: {len(pre):,} across {pre['patient_id'].nunique():,} patients")

    rows = []
    for k, (pid, g) in enumerate(pre.groupby("patient_id")):
        if len(g) < 1: continue
        probs, hrs = hmm_trajectory(g, stats)
        slope = 0.0
        if len(probs) >= 3 and np.ptp(hrs) > 0:
            slope = float(np.polyfit(hrs, probs, 1)[0])
        rows.append(dict(patient_id=pid, hmm_last=float(probs[-1]),
                         hmm_max=float(probs.max()), hmm_mean=float(probs.mean()),
                         hmm_slope=slope, n_pre_windows=len(probs),
                         obs_hours=float(np.ptp(hrs)) if len(hrs) else 0.0))
        if (k+1) % 1000 == 0: print(f"    [{k+1}] patients")
    hmm_df = pd.DataFrame(rows)
    print(f"✅ HMM aggregates for {len(hmm_df):,} patients")
    return hmm_df

POP_STATS = robust_stats(TIMELINE_CSV)
hmm_df    = run_prelandmark_hmm(POP_STATS, cohort)


✅ Robust stats for 16 features
  pre-landmark windows: 18,531 across 3,213 patients
    [1000] patients
    [2000] patients
    [3000] patients
✅ HMM aggregates for 3,213 patients


## Step 3 — Static lab scores (FIB-4, APRI, MELD-Na)

In [4]:
# ============================================================
# STEP 3 — STATIC LAB SCORES from pre-landmark labs (FIB-4, APRI, MELD)
# ============================================================
def prelandmark_lab_features(cohort):
    admit = dict(zip(cohort["patient_id"], pd.to_datetime(cohort["admittime"])))
    pid_set = set(cohort["patient_id"])
    age = dict(zip(cohort["patient_id"], cohort["anchor_age"]))
    need = ["AST","ALT","Platelets","Bilirubin","INR","Creatinine","Sodium",
            "Albumin","MELD_Na"]
    parts = []
    for ch in pd.read_csv(TIMELINE_CSV, chunksize=CHUNK,
                          usecols=["patient_id","timestamp"]+need, low_memory=False):
        ch = ch[ch["patient_id"].isin(pid_set)]
        if ch.empty: continue
        ch["timestamp"] = pd.to_datetime(ch["timestamp"], errors="coerce")
        ch["adm"] = ch["patient_id"].map(admit)
        ch["hrs"] = (ch["timestamp"] - ch["adm"]).dt.total_seconds()/3600
        ch = ch[(ch["hrs"] >= 0) & (ch["hrs"] <= LANDMARK_HOURS)]
        if not ch.empty: parts.append(ch)
    pre = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

    rows = []
    for pid, g in pre.groupby("patient_id"):
        g = g.sort_values("hrs")
        def last(c):   # most recent non-null value before the landmark
            s = g[c].dropna()
            return float(s.iloc[-1]) if len(s) else np.nan
        def worst(c, hi=True):
            s = g[c].dropna()
            if not len(s): return np.nan
            return float(s.max() if hi else s.min())
        ast, alt, plt_ = last("AST"), last("ALT"), last("Platelets")
        a = age.get(pid, np.nan)
        fib4 = (a*ast)/(plt_*np.sqrt(alt)) if all(pd.notna([a,ast,alt,plt_])) and alt>0 and plt_>0 else np.nan
        apri = ((ast/AST_ULN)/plt_)*100 if pd.notna(ast) and pd.notna(plt_) and plt_>0 else np.nan
        astalt = ast/alt if pd.notna(ast) and pd.notna(alt) and alt>0 else np.nan
        rows.append(dict(patient_id=pid,
            bilirubin_last=last("Bilirubin"), albumin_last=last("Albumin"),
            inr_last=last("INR"), creatinine_last=last("Creatinine"),
            sodium_last=last("Sodium"), platelets_last=plt_, ast_last=ast, alt_last=alt,
            fib4=fib4, apri=apri, ast_alt_ratio=astalt,
            meld_last=last("MELD_Na"), meld_max=worst("MELD_Na", hi=True),
            bilirubin_max=worst("Bilirubin", hi=True),
            platelets_min=worst("Platelets", hi=False)))
    lab_df = pd.DataFrame(rows)
    print(f"✅ Lab-score features for {len(lab_df):,} patients "
          f"(FIB-4 non-null: {lab_df['fib4'].notna().sum():,}, "
          f"APRI: {lab_df['apri'].notna().sum():,}, MELD: {lab_df['meld_last'].notna().sum():,})")
    return lab_df

lab_df = prelandmark_lab_features(cohort)


✅ Lab-score features for 3,213 patients (FIB-4 non-null: 3,083, APRI: 3,091, MELD: 3,004)


## Step 3b (optional) — urine output from outputevents

In [5]:
# ============================================================
# STEP 3b (OPTIONAL) — urine output from outputevents (needs MIMIC-IV)
# Run only if you want to add a renal/fluid feature. Safe to skip.
# ============================================================
MIMIC_ROOT = "/Users/sumiyasayeed/Documents/mimic4/mimiciv"   # edit if needed
def add_urine_feature(cohort, mimic_root=MIMIC_ROOT):
    icu = os.path.join(mimic_root, "icu")
    def _csv(d, n):
        for e in (".csv",".csv.gz"):
            if os.path.exists(os.path.join(d, n+e)): return os.path.join(d, n+e)
        return None
    oe, di = _csv(icu,"outputevents"), _csv(icu,"d_items")
    if not oe or not di:
        print("  outputevents/d_items not found — skipping urine feature.")
        return pd.DataFrame(columns=["patient_id","urine_rate_mlph"])
    d = pd.read_csv(di); lab = "label" if "label" in d.columns else d.columns[-1]
    urine_ids = d[d[lab].astype(str).str.contains("urine|foley|void", case=False, na=False)]["itemid"].tolist()
    print(f"  urine itemids: {len(urine_ids)} found")
    admit = dict(zip(cohort["patient_id"], pd.to_datetime(cohort["admittime"])))
    hadm_to_pid = dict(zip(cohort["hadm_id"], cohort["patient_id"]))
    hadm_set = set(cohort["hadm_id"].dropna().astype(int))
    agg = {}
    for ch in pd.read_csv(oe, usecols=["hadm_id","charttime","itemid","value"],
                          chunksize=500_000, low_memory=False):
        ch = ch[ch["hadm_id"].isin(hadm_set) & ch["itemid"].isin(urine_ids)]
        if ch.empty: continue
        ch["pid"] = ch["hadm_id"].map(hadm_to_pid)
        ch["charttime"] = pd.to_datetime(ch["charttime"], errors="coerce")
        ch["adm"] = ch["pid"].map(admit)
        ch["hrs"] = (ch["charttime"]-ch["adm"]).dt.total_seconds()/3600
        ch = ch[(ch["hrs"]>=0)&(ch["hrs"]<=LANDMARK_HOURS)]
        for pid, g in ch.groupby("pid"):
            agg.setdefault(pid, 0.0)
            agg[pid] += float(pd.to_numeric(g["value"], errors="coerce").fillna(0).sum())
    rows = [dict(patient_id=p, urine_rate_mlph=v/max(LANDMARK_HOURS,1)) for p,v in agg.items()]
    out = pd.DataFrame(rows)
    print(f"✅ urine feature for {len(out):,} patients")
    return out

# Toggle: set USE_URINE = True to extract (slow; requires MIMIC-IV).
USE_URINE = False
urine_df = add_urine_feature(cohort) if USE_URINE else pd.DataFrame(columns=["patient_id","urine_rate_mlph"])


## Step 4 — Assemble feature matrix

In [6]:
# ============================================================
# STEP 4 — ASSEMBLE FEATURE MATRIX + LABELS
# ============================================================
feat = cohort[["patient_id","ew_label","anchor_age","gender","hrs_to_marker"]].copy()
feat["gender_M"] = (feat["gender"].astype(str).str.upper().str[0] == "M").astype(int)
feat = feat.merge(hmm_df, on="patient_id", how="left")
feat = feat.merge(lab_df, on="patient_id", how="left")
if len(urine_df):
    feat = feat.merge(urine_df, on="patient_id", how="left")

# require at least the core lab scores present
feat = feat[feat["fib4"].notna() | feat["meld_last"].notna()].copy()
print(f"✅ Feature matrix: {feat.shape} | event rate {feat['ew_label'].mean()*100:.1f}%")

MODEL_FEATURES = ["hmm_last","hmm_max","hmm_mean","hmm_slope",
                  "meld_last","meld_max","fib4","apri","ast_alt_ratio",
                  "bilirubin_last","albumin_last","platelets_last","inr_last",
                  "creatinine_last","sodium_last","anchor_age","gender_M"]
if "urine_rate_mlph" in feat.columns:
    MODEL_FEATURES.append("urine_rate_mlph")
MODEL_FEATURES = [c for c in MODEL_FEATURES if c in feat.columns]
print("Model features:", MODEL_FEATURES)


✅ Feature matrix: (3092, 27) | event rate 31.2%
Model features: ['hmm_last', 'hmm_max', 'hmm_mean', 'hmm_slope', 'meld_last', 'meld_max', 'fib4', 'apri', 'ast_alt_ratio', 'bilirubin_last', 'albumin_last', 'platelets_last', 'inr_last', 'creatinine_last', 'sodium_last', 'anchor_age', 'gender_M']


## Step 5 — Evaluation (held-out test + baselines)

In [7]:
# ============================================================
# STEP 5 — HONEST EVALUATION (held-out test set + baselines)
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

y = feat["ew_label"].astype(int).values
Xtr, Xte, ytr, yte = train_test_split(feat, y, test_size=TEST_FRAC,
                                       stratify=y, random_state=RANDOM_SEED)
print(f"Train {len(Xtr):,} ({ytr.mean()*100:.1f}% events) | "
      f"Test {len(Xte):,} ({yte.mean()*100:.1f}% events)\n")

def boot_ci(yt, ys, n=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed); s = []
    yt = np.asarray(yt); ys = np.asarray(ys)
    for _ in range(n):
        idx = rng.integers(0, len(yt), len(yt))
        if not (1 < yt[idx].sum() < len(idx)-1): continue
        s.append(roc_auc_score(yt[idx], ys[idx]))
    return (np.percentile(s, [2.5, 97.5]) if s else (np.nan, np.nan))

# ---- combined model (impute->scale->logistic), fit on train only ----
imp = SimpleImputer(strategy="median").fit(Xtr[MODEL_FEATURES])
sc  = StandardScaler().fit(imp.transform(Xtr[MODEL_FEATURES]))
def prep(df): return sc.transform(imp.transform(df[MODEL_FEATURES]))
clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(prep(Xtr), ytr)
p_comb = clf.predict_proba(prep(Xte))[:, 1]
auc_c = roc_auc_score(yte, p_comb); lo, hi = boot_ci(yte, p_comb)
print("=== TEST-SET DISCRIMINATION ===")
print(f"  COMBINED model     AUC={auc_c:.3f} [{lo:.3f}-{hi:.3f}]  "
      f"PR-AUC={average_precision_score(yte,p_comb):.3f}  Brier={brier_score_loss(yte,p_comb):.3f}")

# ---- single-feature baselines on the SAME test set ----
print("\n  Baselines (single feature, test set):")
results = [("COMBINED", auc_c, lo, hi)]
for col, name in [("meld_max","MELD-Na (max)"),("meld_last","MELD-Na (last)"),
                  ("fib4","FIB-4"),("apri","APRI"),("ast_alt_ratio","AST/ALT ratio"),
                  ("hmm_max","HMM risk (max)"),("hmm_last","HMM risk (last)"),
                  ("platelets_last","Platelets (last)"),("anchor_age","Age")]:
    if col not in Xte.columns: continue
    m = Xte[col].notna().values
    if m.sum() < 20 or len(np.unique(yte[m])) < 2: continue
    sign = -1.0 if col in ("platelets_last",) else 1.0   # low platelets => higher risk
    a = roc_auc_score(yte[m], sign*Xte[col].values[m]); l, h = boot_ci(yte[m], sign*Xte[col].values[m])
    results.append((name, a, l, h))
    print(f"    {name:<18} AUC={a:.3f} [{l:.3f}-{h:.3f}]  (n={int(m.sum())})")

# ---- coefficients of combined model ----
print("\n  Combined-model standardized coefficients:")
for f, c in sorted(zip(MODEL_FEATURES, clf.coef_[0]), key=lambda t: -abs(t[1])):
    print(f"    {f:<18}{c:+.3f}")

# ---- lead-time among test cases ----
cases = Xte[(yte == 1)]
lead = (cases["hrs_to_marker"] - LANDMARK_HOURS).dropna()
if len(lead):
    print(f"\n  Lead-time (test cases, n={len(lead)}): "
          f"median={lead.median():.1f} h  IQR=[{lead.quantile(.25):.1f}-{lead.quantile(.75):.1f}] h")

# ---- ROC plot ----
plt.figure(figsize=(6,5))
fpr, tpr, _ = roc_curve(yte, p_comb)
plt.plot(fpr, tpr, lw=2, label=f"Combined AUC={auc_c:.3f}")
for col, name in [("meld_max","MELD-Na"),("fib4","FIB-4")]:
    if col in Xte.columns:
        m = Xte[col].notna().values
        if m.sum() > 20 and len(np.unique(yte[m])) > 1:
            f2, t2, _ = roc_curve(yte[m], Xte[col].values[m])
            plt.plot(f2, t2, lw=1.2, ls="--", label=f"{name} AUC={roc_auc_score(yte[m], Xte[col].values[m]):.3f}")
plt.plot([0,1],[0,1],"k:",lw=.8)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title(f"Early-Warning of Decompensation (landmark={LANDMARK_HOURS}h)")
plt.legend(fontsize=9); plt.tight_layout()
plt.savefig(OUT/"roc_early_warning.png", dpi=150); plt.close()

pd.DataFrame(results, columns=["model","auc","ci_lo","ci_hi"]).to_csv(OUT/"early_warning_results.csv", index=False)
feat.to_csv(OUT/"early_warning_features.csv", index=False)
print("\n✅ Saved:", OUT/"early_warning_results.csv", "|", OUT/"roc_early_warning.png")


Train 2,164 (31.2% events) | Test 928 (31.2% events)

=== TEST-SET DISCRIMINATION ===
  COMBINED model     AUC=0.726 [0.693-0.760]  PR-AUC=0.576  Brier=0.211

  Baselines (single feature, test set):
    MELD-Na (max)      AUC=0.696 [0.657-0.736]  (n=903)
    MELD-Na (last)     AUC=0.690 [0.652-0.730]  (n=903)
    FIB-4              AUC=0.577 [0.536-0.615]  (n=925)
    APRI               AUC=0.540 [0.498-0.583]  (n=926)
    AST/ALT ratio      AUC=0.627 [0.586-0.668]  (n=925)
    HMM risk (max)     AUC=0.645 [0.607-0.687]  (n=928)
    HMM risk (last)    AUC=0.610 [0.570-0.650]  (n=928)
    Platelets (last)   AUC=0.502 [0.462-0.547]  (n=928)
    Age                AUC=0.536 [0.492-0.575]  (n=928)

  Combined-model standardized coefficients:
    meld_last         +0.510
    albumin_last      -0.461
    hmm_mean          -0.274
    hmm_last          +0.227
    meld_max          +0.206
    hmm_slope         -0.185
    creatinine_last   -0.153
    hmm_max           +0.100
    fib4            

## Part A — Latent severity stage as an early-warning feature
Fit the latent staging HMM (daily, full stay) to *learn* the stage definitions, then for each
early-warning patient compute the **pre-landmark** stage (strictly from windows in
`[admit, admit+L]` — no leakage) and test whether it improves decompensation prediction
beyond MELD/labs/HMM-risk.

In [8]:
"""Missing-aware diagonal-Gaussian HMM with optional treatment fusion.
Modes:
  baseline   : Gaussian emissions, homogeneous transitions
  emission   : Gaussian + Bernoulli(treatment) emissions, homogeneous transitions
  transition : Gaussian emissions, treatment-conditioned transitions (A_off / A_on)
"""
import numpy as np

VAR_FLOOR = 1e-3

def emission_logB(O, mu, var, u=None, p_treat=None):
    """O: T x D (NaN allowed). returns logB: T x K. Optional Bernoulli treatment emission."""
    T, D = O.shape; K = mu.shape[0]
    mask = ~np.isnan(O)                                  # T x D
    diff = O[:, None, :] - mu[None, :, :]                # T x K x D
    term = -0.5 * (np.log(2*np.pi*var)[None] + (diff**2)/var[None])
    term = np.where(mask[:, None, :], term, 0.0)
    logB = term.sum(axis=2)                              # T x K
    if u is not None and p_treat is not None:
        pt = np.clip(p_treat, 1e-4, 1-1e-4)              # K
        logB = logB + (u[:, None]*np.log(pt)[None] + (1-u[:, None])*np.log(1-pt)[None])
    return logB

def fwd_bwd(logB, pi, A_t):
    """A_t: (T-1) x K x K time-indexed transition matrices. Scaled forward-backward."""
    T, K = logB.shape
    m = logB.max(axis=1); B = np.exp(logB - m[:, None])
    alpha = np.zeros((T, K)); c = np.zeros(T)
    alpha[0] = pi * B[0]; c[0] = alpha[0].sum() + 1e-300; alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A_t[t-1]) * B[t]
        c[t] = alpha[t].sum() + 1e-300; alpha[t] /= c[t]
    beta = np.zeros((T, K)); beta[-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = (A_t[t] @ (B[t+1]*beta[t+1])) / c[t+1]
    gamma = alpha*beta; gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
    loglik = np.sum(np.log(c)) + np.sum(m)
    return alpha, beta, gamma, B, c, loglik

def _init(seqs, K, seed=0):
    rng = np.random.default_rng(seed)
    allobs = np.vstack([s["O"] for s in seqs])
    D = allobs.shape[1]
    med = np.nanmedian(allobs, axis=0)
    # severity proxy = mean of standardized obs (already standardized upstream) -> quantile init
    sev = np.nanmean(allobs, axis=1)
    qs = np.nanquantile(sev, np.linspace(0.1, 0.9, K))
    mu = np.tile(med, (K, 1)).astype(float)
    # spread state means along each dim by quantiles of that dim
    for d in range(D):
        col = allobs[:, d]; col = col[~np.isnan(col)]
        if len(col):
            mu[:, d] = np.quantile(col, np.linspace(0.2, 0.8, K))
    var = np.tile(np.nanvar(allobs, axis=0), (K, 1)); var[var < VAR_FLOOR] = 1.0
    pi = np.full(K, 1.0/K)
    A = np.full((K, K), 0.1/(K-1)) + np.eye(K)*0.9
    A /= A.sum(axis=1, keepdims=True)
    return pi, A, mu, var

def fit(seqs, K=4, mode="baseline", n_iter=25, seed=0, verbose=False):
    """seqs: list of dict with 'O' (T x D) and optional 'u' (T,) treatment binary."""
    pi, A, mu, var = _init(seqs, K, seed)
    A_on = A.copy(); A_off = A.copy()
    p_treat = np.linspace(0.2, 0.8, K) if mode == "emission" else None
    D = mu.shape[1]
    prev_ll = -np.inf
    for it in range(n_iter):
        pi_acc = np.zeros(K)
        num_mu = np.zeros((K, D)); den_mu = np.zeros((K, D)); num_var = np.zeros((K, D))
        if mode == "transition":
            xi_off = np.zeros((K, K)); xi_on = np.zeros((K, K))
        else:
            xi_sum = np.zeros((K, K))
        tp_num = np.zeros(K); tp_den = np.zeros(K)
        total_ll = 0.0
        for s in seqs:
            O = s["O"]; T = O.shape[0]
            u = s.get("u") if mode != "baseline" else None
            logB = emission_logB(O, mu, var, u=(u if mode == "emission" else None),
                                 p_treat=(p_treat if mode == "emission" else None))
            if mode == "transition" and u is not None:
                A_t = np.stack([A_on if u[t] > 0.5 else A_off for t in range(T-1)]) if T > 1 \
                      else np.zeros((0, K, K))
            else:
                A_t = np.stack([A]*(T-1)) if T > 1 else np.zeros((0, K, K))
            alpha, beta, gamma, B, c, ll = fwd_bwd(logB, pi, A_t)
            total_ll += ll
            pi_acc += gamma[0]
            mask = ~np.isnan(O)
            # mean AND variance from the SAME posteriors (cached gamma)
            Osq = np.where(mask, O*O, 0.0)
            for k in range(K):
                g = gamma[:, k]
                num_mu[k]  += np.nansum(np.where(mask, O, 0.0) * g[:, None], axis=0)
                den_mu[k]  += (mask * g[:, None]).sum(axis=0)
                num_var[k] += (Osq * g[:, None]).sum(axis=0)   # sum gamma*x^2 (suff. stat)
            for t in range(T-1):
                At = (A_on if (mode == "transition" and u is not None and u[t] > 0.5)
                      else A_off if mode == "transition" else A)
                xi = (alpha[t][:, None] * At * (B[t+1]*beta[t+1])[None, :]) / (c[t+1])
                xi /= xi.sum() + 1e-300
                if mode == "transition" and u is not None:
                    if u[t] > 0.5: xi_on += xi
                    else: xi_off += xi
                else:
                    xi_sum += xi
            if mode == "emission" and u is not None:
                tp_num += (gamma * u[:, None]).sum(axis=0)
                tp_den += gamma.sum(axis=0)
        # M-step
        N = len(seqs)
        pi = pi_acc / N; pi /= pi.sum()
        mu = num_mu / np.maximum(den_mu, 1e-6)
        var = num_var / np.maximum(den_mu, 1e-6) - mu**2   # E[x^2]-E[x]^2
        var[var < VAR_FLOOR] = VAR_FLOOR
        if mode == "transition":
            A_off = xi_off / np.maximum(xi_off.sum(axis=1, keepdims=True), 1e-12)
            A_on  = xi_on  / np.maximum(xi_on.sum(axis=1, keepdims=True), 1e-12)
        else:
            A = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-12)
        if mode == "emission":
            p_treat = np.clip(tp_num/np.maximum(tp_den, 1e-6), 1e-4, 1-1e-4)
        if verbose:
            print(f"  iter {it+1}: loglik={total_ll:.1f}")
        if abs(total_ll - prev_ll) < 1e-3*abs(prev_ll+1e-9):
            break
        prev_ll = total_ll
    params = dict(pi=pi, mu=mu, var=var, K=K, mode=mode)
    if mode == "transition": params["A_off"], params["A_on"] = A_off, A_on
    else: params["A"] = A
    if mode == "emission": params["p_treat"] = p_treat
    params["loglik"] = total_ll
    return params

def reorder_by_severity(params, direction):
    """direction: array length D, sign of severity per obs dim. Sort states ascending severity."""
    mu = params["mu"]; sev = (mu * direction[None, :]).sum(axis=1)
    order = np.argsort(sev)
    inv = np.argsort(order)
    p = dict(params)
    p["mu"] = mu[order]; p["var"] = params["var"][order]; p["pi"] = params["pi"][order]
    p["pi"] /= p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order, order)]
        p["A_on"]  = params["A_on"][np.ix_(order, order)]
    else:
        p["A"] = params["A"][np.ix_(order, order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    p["_order"] = order
    return p

def posterior_stage(params, O, u=None):
    """Return gamma (T x K) and expected stage per window."""
    mode = params["mode"]; T = O.shape[0]
    logB = emission_logB(O, params["mu"], params["var"],
                         u=(u if mode == "emission" else None),
                         p_treat=(params.get("p_treat") if mode == "emission" else None))
    if mode == "transition" and u is not None:
        A_t = np.stack([params["A_on"] if u[t] > 0.5 else params["A_off"] for t in range(T-1)]) if T>1 else np.zeros((0,params["K"],params["K"]))
    else:
        A = params.get("A", params.get("A_off"))
        A_t = np.stack([A]*(T-1)) if T>1 else np.zeros((0,params["K"],params["K"]))
    _,_,gamma,_,_,ll = fwd_bwd(logB, params["pi"], A_t)
    stages = (gamma * np.arange(params["K"])[None, :]).sum(axis=1)
    return gamma, stages, ll

def obs_pred_loglik(params, seqs):
    """FAIR cross-mode metric: one-step-ahead predictive log-lik of the next
    observation vector, P(O_{t+1} | O_{1:t}, u_{1:t}). Gaussian (lab) part only,
    so emission/transition/baseline are scored on the SAME target."""
    mode = params["mode"]; K = params["K"]; mu = params["mu"]; var = params["var"]
    A_def = params.get("A", params.get("A_off"))
    tot = 0.0; nwin = 0
    for s in seqs:
        O = s["O"]; T = O.shape[0]; u = s.get("u")
        logB = emission_logB(O, mu, var,
                             u=(u if mode == "emission" else None),
                             p_treat=(params.get("p_treat") if mode == "emission" else None))
        m = logB.max(axis=1); B = np.exp(logB - m[:, None])      # full emission for filtering
        gaussB = np.exp(emission_logB(O, mu, var) -              # gaussian-only for prediction
                        emission_logB(O, mu, var).max(axis=1)[:, None])
        gm = emission_logB(O, mu, var).max(axis=1)
        alpha = params["pi"] * B[0]; alpha /= alpha.sum() + 1e-300
        for t in range(T-1):
            if mode == "transition" and u is not None:
                A = params["A_on"] if u[t] > 0.5 else params["A_off"]
            else:
                A = A_def
            pred = alpha @ A
            pdens = (pred * gaussB[t+1]).sum()                   # predictive density (scaled)
            tot += np.log(pdens + 1e-300) + gm[t+1]              # undo gauss scaling
            nwin += 1
            alpha = pred * B[t+1]; alpha /= alpha.sum() + 1e-300
    return tot, nwin, tot/max(nwin, 1)

def heldout_loglik(params, seqs):
    tot = 0.0; nwin = 0
    for s in seqs:
        u = s.get("u")
        _,_,ll = posterior_stage(params, s["O"], u=u)
        tot += ll; nwin += s["O"].shape[0]
    return tot, nwin, tot/max(nwin,1)


In [9]:
# ---- Fit latent staging HMM on daily, full-stay data (learns stage definitions) ----
OBS_STAGE = ["Bilirubin","INR","Albumin","Creatinine","Sodium","MAP","Ammonia","Lactate"]
K_STAGE   = 3
_df = pd.read_csv(TIMELINE_CSV, usecols=["patient_id","timestamp"]+OBS_STAGE+["MELD_Na"], low_memory=False)
_df["timestamp"] = pd.to_datetime(_df["timestamp"], errors="coerce")
_df["day"] = _df["timestamp"].dt.floor("1D")
_daily = (_df.groupby(["patient_id","day"])[OBS_STAGE+["MELD_Na"]].median()
            .reset_index().sort_values(["patient_id","day"]))
_daily[OBS_STAGE] = _daily.groupby("patient_id")[OBS_STAGE].ffill()
STD_STAGE = {}
for c in OBS_STAGE:
    v = _daily[c].dropna().values; med = float(np.median(v))
    iqr = float(np.percentile(v,75)-np.percentile(v,25)); sd = iqr/1.349 if iqr>1e-6 else float(np.std(v))
    sd = sd if sd > 1e-6 else 1.0; STD_STAGE[c] = (med, sd)
    _daily[c] = np.clip((_daily[c]-med)/sd, -8, 8)
_seqs = []
for pid, g in _daily.groupby("patient_id"):
    O = g[OBS_STAGE].values.astype(float)
    if len(O) < 2: continue
    _seqs.append(dict(pid=pid, O=O, ts=g["day"].values, meld=g["MELD_Na"].values))
stage_params = fit(_seqs, K=K_STAGE, mode="baseline", n_iter=25, seed=1)
# relabel stages by posterior-weighted MELD (Stage 0 = least severe)
K = K_STAGE; num = np.zeros(K); den = np.zeros(K)
for s in _seqs:
    g,_,_ = posterior_stage(stage_params, s["O"]); m = ~np.isnan(s["meld"])
    if m.any(): num += (g[m]*s["meld"][m,None]).sum(0); den += g[m].sum(0)
_ord = np.argsort(num/np.maximum(den,1e-6))
stage_params = dict(stage_params, mu=stage_params["mu"][_ord], var=stage_params["var"][_ord],
                    pi=stage_params["pi"][_ord]/stage_params["pi"][_ord].sum(),
                    A=stage_params["A"][np.ix_(_ord,_ord)])
print(f"✅ staging HMM fit (K={K_STAGE}) on {len(_seqs):,} patients; stages MELD-ordered")


✅ staging HMM fit (K=3) on 7,503 patients; stages MELD-ordered


In [10]:
# ---- Pre-landmark latent stage per early-warning patient (time-censored) ----
admit = dict(zip(cohort["patient_id"], pd.to_datetime(cohort["admittime"])))
pid_set = set(cohort["patient_id"])
parts = []
for ch in pd.read_csv(TIMELINE_CSV, chunksize=CHUNK,
                      usecols=["patient_id","timestamp"]+OBS_STAGE, low_memory=False):
    ch = ch[ch["patient_id"].isin(pid_set)]
    if ch.empty: continue
    ch["timestamp"] = pd.to_datetime(ch["timestamp"], errors="coerce")
    ch["adm"] = ch["patient_id"].map(admit)
    ch["hrs"] = (ch["timestamp"]-ch["adm"]).dt.total_seconds()/3600
    ch = ch[(ch["hrs"] >= 0) & (ch["hrs"] <= LANDMARK_HOURS)]
    if not ch.empty: parts.append(ch)
pre = pd.concat(parts, ignore_index=True)
for c in OBS_STAGE:
    med, sd = STD_STAGE[c]; pre[c] = np.clip((pre[c]-med)/sd, -8, 8)
rows = []
for pid, g in pre.sort_values("hrs").groupby("patient_id"):
    O = g[OBS_STAGE].values.astype(float)
    if len(O) == 0: continue
    gp, stages, _ = posterior_stage(stage_params, O)
    rows.append(dict(patient_id=pid, ew_stage_last=float(stages[-1]),
                     ew_stage_peak=float(stages.max()), ew_p_top=float(gp[-1, K_STAGE-1])))
stage_feat = pd.DataFrame(rows)
feat = feat.merge(stage_feat, on="patient_id", how="left")
print(f"✅ pre-landmark stage features for {len(stage_feat):,} patients merged into feat")


✅ pre-landmark stage features for 3,213 patients merged into feat


In [11]:
# ---- Does the latent stage improve early-warning beyond MELD/labs/HMM-risk? ----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

STAGE_FEATS = ["ew_stage_last","ew_stage_peak","ew_p_top"]
y = feat["ew_label"].astype(int).values
Xtr, Xte, ytr, yte = train_test_split(feat, y, test_size=TEST_FRAC, stratify=y, random_state=RANDOM_SEED)

def _auc(features):
    fs = [c for c in features if c in feat.columns]
    imp = SimpleImputer(strategy="median").fit(Xtr[fs]); sc = StandardScaler().fit(imp.transform(Xtr[fs]))
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(imp.transform(Xtr[fs])), ytr)
    p = clf.predict_proba(sc.transform(imp.transform(Xte[fs])))[:,1]
    return roc_auc_score(yte, p), p

def _bdiff(p1, p2, n=1000, seed=42):
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        idx = rng.integers(0, len(yte), len(yte))
        if not (1 < yte[idx].sum() < len(idx)-1): continue
        d.append(roc_auc_score(yte[idx], p2[idx]) - roc_auc_score(yte[idx], p1[idx]))
    return np.percentile(d, [2.5, 97.5])

auc_base, p_base   = _auc(MODEL_FEATURES)
auc_stage, p_stage = _auc(MODEL_FEATURES + STAGE_FEATS)
auc_only, _        = _auc(STAGE_FEATS)
lo, hi = _bdiff(p_base, p_stage)
print(f"WITHOUT stage : AUC={auc_base:.3f}")
print(f"WITH stage    : AUC={auc_stage:.3f}  (delta {auc_stage-auc_base:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}])")
print(f"Stage ALONE   : AUC={auc_only:.3f}")
print("Stage adds signal beyond the existing features ONLY if the delta CI excludes 0.")


WITHOUT stage : AUC=0.726
WITH stage    : AUC=0.729  (delta +0.002, 95% CI [-0.004, +0.009])
Stage ALONE   : AUC=0.633
Stage adds signal beyond the existing features ONLY if the delta CI excludes 0.


## Part B — Causal transition-fusion (IPW-adjusted marginal structural multistate model)
The v2 fusion showed treated patients progress to sicker stages far more often — but that is
**confounding by indication** (rescue drugs go to deteriorating patients). Here we test whether
the treatment→progression association *survives adjustment for severity*:
fit a propensity model `P(treatment | current stage + labs)`, form **stabilized IPW weights**,
and re-estimate the off/on stage-progression rates in the weighted pseudo-population.
If the gap collapses after weighting → it was confounding. If it persists → a candidate
(still associational, but adjusted) treatment effect on dynamics.

*Requires MIMIC-IV (prescriptions + inputevents). Associational-with-adjustment, not a randomized causal claim.*

In [12]:
# ---- Treatment extraction per class (daily), attached to staging sequences ----
MIMIC_ROOT = "/Users/sumiyasayeed/Documents/mimic4/mimiciv"
DRUG_REGEX = {"lactulose": r"lactulose|rifaximin",
              "diuretic":  r"furosemide|spironolactone|bumetanide|torsemide"}
INPUT_ITEMIDS = {"vasopressor": [221906,221289,221749,222315,221662,221653],
                 "albumin":     [220864,220862]}
def _csv(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return None
def build_intervals():
    hosp, icu = os.path.join(MIMIC_ROOT,"hosp"), os.path.join(MIMIC_ROOT,"icu")
    co = pd.read_csv(COHORT_CSV, usecols=["hadm_id","patient_id"])
    hadm_pid = dict(zip(co.hadm_id, co.patient_id)); hadm_set = set(co.hadm_id.dropna().astype(int))
    cls = {k: {} for k in list(DRUG_REGEX)+list(INPUT_ITEMIDS)}
    pres = _csv(hosp,"prescriptions")
    if pres:
        for ch in pd.read_csv(pres, usecols=["hadm_id","drug","starttime","stoptime"], chunksize=500_000, low_memory=False):
            ch = ch[ch.hadm_id.isin(hadm_set)]
            if ch.empty: continue
            dl = ch["drug"].astype(str).str.lower()
            for name, rx in DRUG_REGEX.items():
                for _, r in ch[dl.str.contains(rx, regex=True, na=False)].iterrows():
                    pid = hadm_pid.get(r.hadm_id); s = pd.to_datetime(r.starttime, errors="coerce")
                    if pid is None or pd.isna(s): continue
                    e = pd.to_datetime(r.stoptime, errors="coerce"); e = e if pd.notna(e) else s+pd.Timedelta("24h")
                    cls[name].setdefault(pid, []).append((s, e))
    inp = _csv(icu,"inputevents")
    if inp:
        for ch in pd.read_csv(inp, usecols=["hadm_id","itemid","starttime","endtime"], chunksize=500_000, low_memory=False):
            ch = ch[ch.hadm_id.isin(hadm_set)]
            if ch.empty: continue
            for name, ids in INPUT_ITEMIDS.items():
                for _, r in ch[ch.itemid.isin(ids)].iterrows():
                    pid = hadm_pid.get(r.hadm_id); s = pd.to_datetime(r.starttime, errors="coerce")
                    if pid is None or pd.isna(s): continue
                    e = pd.to_datetime(r.endtime, errors="coerce"); e = e if pd.notna(e) else s+pd.Timedelta("1h")
                    cls[name].setdefault(pid, []).append((s, e))
    return cls
def attach(seqs, intervals, key, win="1D"):
    w = pd.Timedelta(win)
    for s in seqs:
        ts = pd.to_datetime(s["ts"]); u = np.zeros(len(ts))
        for (st, en) in intervals.get(s["pid"], []):
            u[np.asarray((ts < en) & ((ts+w) > st))] = 1.0
        s["u_"+key] = u

if os.path.exists(os.path.join(MIMIC_ROOT,"hosp")):
    CLS = build_intervals()
    for k in CLS: attach(_seqs, CLS[k], k)
    TREAT_CLASSES = list(CLS)
    print("✅ treatments attached:", TREAT_CLASSES)
else:
    TREAT_CLASSES = []
    print("MIMIC-IV not found — Part B skipped.")


✅ treatments attached: ['lactulose', 'diuretic', 'vasopressor', 'albumin']


In [13]:
# ---- IPW marginal structural multistate transition analysis ----
from sklearn.linear_model import LogisticRegression

def window_table(seqs, ukey):
    rows = []
    for s in seqs:
        if ukey not in s: continue
        g, _, _ = posterior_stage(stage_params, s["O"])
        st = np.argmax(g, axis=1); exp = (g * np.arange(K_STAGE)).sum(axis=1)
        u = s[ukey]; O = s["O"]
        for t in range(len(st)-1):
            rows.append(dict(frm=int(st[t]), to=int(st[t+1]), u=float(u[t]), exp_stage=float(exp[t]),
                             bili=O[t,0], inr=O[t,1], alb=O[t,2], cr=O[t,3], na=O[t,4], map_=O[t,5]))
    return pd.DataFrame(rows)

def causal_fusion(seqs, ukey):
    T = window_table(seqs, ukey)
    if len(T) < 500 or not (0.02 < T["u"].mean() < 0.6):
        print(f"[{ukey}] skipped (prevalence {T['u'].mean()*100:.1f}%)"); return
    conf = ["exp_stage","bili","inr","alb","cr","na","map_"]
    T = T.dropna(subset=conf).copy()
    ps = LogisticRegression(max_iter=1000).fit(T[conf], T["u"]).predict_proba(T[conf])[:,1]
    pm = T["u"].mean()
    T["sw"] = np.clip(np.where(T["u"] > 0.5, pm/np.clip(ps,1e-3,1-1e-3),
                               (1-pm)/np.clip(1-ps,1e-3,1-1e-3)), 0.1, 10)
    T["prog"] = (T["to"] > T["frm"]).astype(int)
    on, off = T[T.u > 0.5], T[T.u < 0.5]
    un = on["prog"].mean() - off["prog"].mean()
    wn = np.average(on["prog"], weights=on["sw"]); wf = np.average(off["prog"], weights=off["sw"])
    wg = wn - wf
    print(f"[{ukey}] windows={len(T):,} treated={int(T.u.sum()):,}")
    print(f"   UNADJUSTED  : P(progress) off={off['prog'].mean():.3f} on={on['prog'].mean():.3f} | gap={un:+.3f}")
    print(f"   IPW-ADJUSTED: P(progress) off={wf:.3f} on={wn:.3f} | gap={wg:+.3f}")
    if abs(un) < 0.005:
        print("   baseline off/on gap is negligible -> no confounding signal to adjust")
    else:
        shrink = (1 - wg/un)*100
        tag = ("mostly CONFOUNDING" if shrink > 50 else
               "partly persists" if shrink > 0 else
               "persists or grows (adjustment did not shrink the gap)")
        print(f"   gap shrinks {shrink:.0f}% after severity adjustment  ->  {tag}")

for key in TREAT_CLASSES:
    causal_fusion(_seqs, "u_"+key)


[u_lactulose] windows=31,310 treated=17,739
   UNADJUSTED  : P(progress) off=0.016 on=0.013 | gap=-0.003
   IPW-ADJUSTED: P(progress) off=0.013 on=0.014 | gap=+0.001
   baseline off/on gap is negligible -> no confounding signal to adjust
[u_diuretic] windows=31,310 treated=11,009
   UNADJUSTED  : P(progress) off=0.013 on=0.016 | gap=+0.003
   IPW-ADJUSTED: P(progress) off=0.013 on=0.015 | gap=+0.001
   baseline off/on gap is negligible -> no confounding signal to adjust
[u_vasopressor] windows=31,310 treated=5,226
   UNADJUSTED  : P(progress) off=0.013 on=0.018 | gap=+0.005
   IPW-ADJUSTED: P(progress) off=0.013 on=0.023 | gap=+0.010
   baseline off/on gap is negligible -> no confounding signal to adjust
[u_albumin] windows=31,310 treated=2,787
   UNADJUSTED  : P(progress) off=0.013 on=0.022 | gap=+0.008
   IPW-ADJUSTED: P(progress) off=0.013 on=0.024 | gap=+0.011
   gap shrinks -35% after severity adjustment  ->  persists or grows (adjustment did not shrink the gap)


## Part C — Lifting performance with orthogonal signal
Two feature blocks that add information **MELD does not contain**:
- **Prior decompensation history** (strictly from admissions *before* the index admission — no
  leakage): chronicity is the strongest non-lab predictor of future decompensation.
- **Trajectory / velocity** over the first 24 h (slopes, variability, deltas of key labs):
  deterioration *rate*, orthogonal to any single MELD snapshot.
Re-evaluated on the same held-out test split, with bootstrap CIs on the AUC gain.

In [14]:
# ---- Prior decompensation history (MIMIC; strictly pre-index admission) ----
MIMIC_ROOT = globals().get("MIMIC_ROOT", "/Users/sumiyasayeed/Documents/mimic4/mimiciv")
def _csvp(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return None
CIRR_PREF  = {9:("5712","5715","5716"),
              10:("K7030","K7031","K717","K740","K741","K742","K743","K744","K745","K746")}
DECOMP_PREF= {9:("78959","7895","4560","45620","5722","56723","5724"),
              10:("R180","R188","R189","I8501","I8511","K720","K721","K729","K7290","K7291","K652","K767")}
def _pref(series, p9, p10, ver):
    s = series.astype(str).str.replace(".","",regex=False).str.upper()
    return ((ver==9)&s.str.startswith(p9)) | ((ver==10)&s.str.startswith(p10))

def prior_history():
    hosp = os.path.join(MIMIC_ROOT,"hosp")
    cocsv = pd.read_csv(COHORT_CSV, usecols=["subject_id","patient_id","admittime"])
    cocsv["admittime"] = pd.to_datetime(cocsv["admittime"], errors="coerce")
    idx_admit = dict(zip(cocsv.subject_id, cocsv.admittime))
    subj = set(cocsv.subject_id)
    adm = pd.read_csv(_csvp(hosp,"admissions"), usecols=["subject_id","hadm_id","admittime"])
    adm = adm[adm.subject_id.isin(subj)].copy()
    adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
    adm["idx_admit"] = adm.subject_id.map(idx_admit)
    prior = adm[adm.admittime < adm.idx_admit].copy()          # strictly before index
    # flag decomp / cirrhosis on prior admissions
    dx = pd.read_csv(_csvp(hosp,"diagnoses_icd"), usecols=["subject_id","hadm_id","icd_code","icd_version"])
    dx = dx[dx.subject_id.isin(subj)]
    dx["dec"] = _pref(dx.icd_code, DECOMP_PREF[9], DECOMP_PREF[10], dx.icd_version).astype(int)
    dx["cir"] = _pref(dx.icd_code, CIRR_PREF[9], CIRR_PREF[10], dx.icd_version).astype(int)
    perh = dx.groupby("hadm_id")[["dec","cir"]].max().reset_index()
    prior = prior.merge(perh, on="hadm_id", how="left").fillna({"dec":0,"cir":0})
    g = prior.groupby("subject_id")
    hist = pd.DataFrame({
        "n_prior_adm":        g.size(),
        "prior_decomp_count": g["dec"].sum(),
        "prior_cirr_adm":     g["cir"].sum(),
        "last_prior_admit":   g["admittime"].max(),
    }).reset_index()
    hist = cocsv.merge(hist, on="subject_id", how="left")
    hist["n_prior_adm"]        = hist["n_prior_adm"].fillna(0)
    hist["prior_decomp_count"] = hist["prior_decomp_count"].fillna(0)
    hist["prior_cirr_adm"]     = hist["prior_cirr_adm"].fillna(0)
    hist["prior_decomp"]       = (hist["prior_decomp_count"] > 0).astype(int)
    hist["days_since_last_adm"]= (hist["admittime"]-hist["last_prior_admit"]).dt.total_seconds()/86400
    hist["days_since_last_adm"]= hist["days_since_last_adm"].fillna(9999)   # no prior admission
    return hist[["patient_id","n_prior_adm","prior_decomp","prior_decomp_count",
                 "prior_cirr_adm","days_since_last_adm"]]

HIST_FEATS = ["n_prior_adm","prior_decomp","prior_decomp_count","prior_cirr_adm","days_since_last_adm"]
if os.path.exists(os.path.join(MIMIC_ROOT,"hosp")):
    hist_feat = prior_history()
    feat = feat.merge(hist_feat, on="patient_id", how="left")
    print(f"✅ prior-history features merged | prior_decomp rate: "
          f"{feat['prior_decomp'].mean()*100:.1f}% | median prior adm: {feat['n_prior_adm'].median():.0f}")
else:
    HIST_FEATS = []
    print("MIMIC-IV not found — prior-history skipped.")


✅ prior-history features merged | prior_decomp rate: 2.1% | median prior adm: 0


In [15]:
# ---- Trajectory / velocity features (pre-landmark; from timeline) ----
TRAJ_LABS = ["Bilirubin","Creatinine","INR","WBC","Sodium","MAP","Lactate","Albumin"]
admit = dict(zip(pd.read_csv(COHORT_CSV, usecols=["patient_id","admittime"]).patient_id,
                 pd.to_datetime(pd.read_csv(COHORT_CSV, usecols=["patient_id","admittime"]).admittime)))
pid_set = set(feat["patient_id"])
parts = []
for ch in pd.read_csv(TIMELINE_CSV, chunksize=CHUNK,
                      usecols=["patient_id","timestamp"]+TRAJ_LABS, low_memory=False):
    ch = ch[ch.patient_id.isin(pid_set)]
    if ch.empty: continue
    ch["timestamp"] = pd.to_datetime(ch["timestamp"], errors="coerce")
    ch["adm"] = ch.patient_id.map(admit)
    ch["hrs"] = (ch["timestamp"]-ch["adm"]).dt.total_seconds()/3600
    ch = ch[(ch["hrs"]>=0)&(ch["hrs"]<=LANDMARK_HOURS)]
    if not ch.empty: parts.append(ch)
pre = pd.concat(parts, ignore_index=True)

def _slope(h, v):
    m = ~np.isnan(v)
    if m.sum() < 2 or np.ptp(h[m]) < 1e-6: return 0.0
    return float(np.polyfit(h[m], v[m], 1)[0])     # per-hour change
rows = []
for pid, g in pre.sort_values("hrs").groupby("patient_id"):
    h = g["hrs"].values; r = {"patient_id": pid}
    for c in ["Bilirubin","Creatinine","INR","WBC","MAP"]:
        r["slope_"+c.lower()] = _slope(h, g[c].values.astype(float))
    for c in ["Creatinine","MAP","Lactate"]:
        v = g[c].dropna().values; r["std_"+c.lower()] = float(np.std(v)) if len(v) > 1 else 0.0
    for c in ["Bilirubin","Creatinine"]:
        v = g[c].dropna().values
        r["delta_"+c.lower()] = float(v[-1]-v[0]) if len(v) >= 2 else 0.0
    rows.append(r)
traj_feat = pd.DataFrame(rows)
feat = feat.merge(traj_feat, on="patient_id", how="left")
TRAJ_FEATS = [c for c in traj_feat.columns if c != "patient_id"]
# optional urine (if Step 3b was run / available)
if "urine_rate_mlph" in feat.columns and "urine_rate_mlph" not in TRAJ_FEATS:
    TRAJ_FEATS = TRAJ_FEATS + ["urine_rate_mlph"]
print(f"✅ trajectory features ({len(TRAJ_FEATS)}) merged: {TRAJ_FEATS}")


✅ trajectory features (10) merged: ['slope_bilirubin', 'slope_creatinine', 'slope_inr', 'slope_wbc', 'slope_map', 'std_creatinine', 'std_map', 'std_lactate', 'delta_bilirubin', 'delta_creatinine']


In [16]:
# ---- Re-evaluate: how much do history + trajectory lift over the base model? ----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

y = feat["ew_label"].astype(int).values
Xtr, Xte, ytr, yte = train_test_split(feat, y, test_size=TEST_FRAC, stratify=y, random_state=RANDOM_SEED)
def auc_of(features):
    fs = [c for c in features if c in feat.columns]
    imp = SimpleImputer(strategy="median").fit(Xtr[fs]); sc = StandardScaler().fit(imp.transform(Xtr[fs]))
    clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(sc.transform(imp.transform(Xtr[fs])), ytr)
    p = clf.predict_proba(sc.transform(imp.transform(Xte[fs])))[:,1]
    return roc_auc_score(yte, p), average_precision_score(yte, p), p
def bdiff(p1, p2, n=2000, seed=42):
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        idx = rng.integers(0, len(yte), len(yte))
        if not (1 < yte[idx].sum() < len(idx)-1): continue
        d.append(roc_auc_score(yte[idx], p2[idx]) - roc_auc_score(yte[idx], p1[idx]))
    return float(np.mean(d)), np.percentile(d, [2.5, 97.5])

base = MODEL_FEATURES
blocks = {"base": base, "base+history": base+HIST_FEATS,
          "base+trajectory": base+TRAJ_FEATS, "base+history+trajectory": base+HIST_FEATS+TRAJ_FEATS}
aucs = {}; ps = {}
for name, fl in blocks.items():
    a, pr, p = auc_of(fl); aucs[name] = (a, pr); ps[name] = p
    print(f"  {name:<26} AUC={a:.3f}  PR-AUC={pr:.3f}")
print()
for name in ["base+history","base+trajectory","base+history+trajectory"]:
    md, ci = bdiff(ps["base"], ps[name])
    sig = "**" if (ci[0] > 0 or ci[1] < 0) else "ns"
    print(f"  delta vs base: {name:<26} {md:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  {sig}")
print("\n  (MELD-Na baseline on this split was ~0.696; base model ~0.726.)")


  base                       AUC=0.726  PR-AUC=0.576
  base+history               AUC=0.728  PR-AUC=0.579
  base+trajectory            AUC=0.729  PR-AUC=0.582
  base+history+trajectory    AUC=0.729  PR-AUC=0.584

  delta vs base: base+history               +0.002  95% CI [-0.002, +0.006]  ns
  delta vs base: base+trajectory            +0.002  95% CI [-0.007, +0.011]  ns
  delta vs base: base+history+trajectory    +0.003  95% CI [-0.007, +0.012]  ns

  (MELD-Na baseline on this split was ~0.696; base model ~0.726.)


## Part D — Subtype-specific early-warning models
The composite outcome lumps physiologically distinct events. Here we build a **separate
early-warning model per decompensation subtype**, using subtype-specific *timestamped* markers:
- **Hepatic encephalopathy (HE)** — lactulose / rifaximin start
- **Variceal bleeding** — octreotide / terlipressin start
- **Ascites** — paracentesis

For each subtype: cases = patients with an *incident* subtype marker (after the landmark);
controls = patients who never decompensate (any type); patients who decompensate only with a
*different* subtype are excluded (kept out of both cases and controls). Same pre-landmark
features and held-out split as the composite model. Reported against the composite (~0.726)
and the per-subtype MELD-Na baseline.

*Constraint: restricted to patients already in the composite feature matrix (so features exist);
SBP/HRS omitted — their markers are low-specificity. Requires MIMIC for marker extraction.*

In [17]:
# ---- Per-subtype timestamped markers (MIMIC) ----
MIMIC_ROOT = globals().get("MIMIC_ROOT", "/Users/sumiyasayeed/Documents/mimic4/mimiciv")
def _csvp(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return None
SUBTYPE_DRUGS = {"HE": r"lactulose|rifaximin", "variceal": r"octreotide|terlipressin"}
PARACENTESIS_RX = r"paracentesis"

def subtype_markers():
    hosp, icu = os.path.join(MIMIC_ROOT,"hosp"), os.path.join(MIMIC_ROOT,"icu")
    co = pd.read_csv(COHORT_CSV, usecols=["subject_id","hadm_id","patient_id"])
    hadm_pid = dict(zip(co.hadm_id, co.patient_id)); hadm_set = set(co.hadm_id.dropna().astype(int))
    mk = {k: {} for k in list(SUBTYPE_DRUGS)+["ascites"]}
    pres = _csvp(hosp,"prescriptions")
    if pres:
        for ch in pd.read_csv(pres, usecols=["hadm_id","drug","starttime"], chunksize=500_000, low_memory=False):
            ch = ch[ch.hadm_id.isin(hadm_set)]
            if ch.empty: continue
            dl = ch["drug"].astype(str).str.lower(); st = pd.to_datetime(ch["starttime"], errors="coerce")
            for name, rx in SUBTYPE_DRUGS.items():
                sub = ch[dl.str.contains(rx, regex=True, na=False)]
                for h, t in zip(sub["hadm_id"], pd.to_datetime(sub["starttime"], errors="coerce")):
                    pid = hadm_pid.get(h)
                    if pid is None or pd.isna(t): continue
                    mk[name][pid] = min(mk[name].get(pid, t), t)
    proc, di = _csvp(icu,"procedureevents"), _csvp(icu,"d_items")
    if proc and di:
        d = pd.read_csv(di); lab = "label" if "label" in d.columns else d.columns[-1]
        ids = d[d[lab].astype(str).str.contains(PARACENTESIS_RX, case=False, na=False)]["itemid"].tolist()
        pe = pd.read_csv(proc, usecols=["hadm_id","itemid","starttime"])
        pe = pe[pe.hadm_id.isin(hadm_set) & pe.itemid.isin(ids)]
        for h, t in zip(pe["hadm_id"], pd.to_datetime(pe["starttime"], errors="coerce")):
            pid = hadm_pid.get(h)
            if pid is None or pd.isna(t): continue
            mk["ascites"][pid] = min(mk["ascites"].get(pid, t), t)
    rows = []
    allpids = set().union(*[set(v) for v in mk.values()])
    for pid in allpids:
        rows.append(dict(patient_id=pid, **{f"{k}_time": mk[k].get(pid) for k in mk}))
    out = pd.DataFrame(rows)
    for k in mk: out[f"{k}_time"] = pd.to_datetime(out[f"{k}_time"])
    print("✅ subtype markers:", {k: int(out[f'{k}_time'].notna().sum()) for k in mk})
    return out

SUBTYPES = ["HE","variceal","ascites"]
if os.path.exists(os.path.join(MIMIC_ROOT,"hosp")):
    submk = subtype_markers()
else:
    submk = pd.DataFrame(columns=["patient_id"]+[f"{s}_time" for s in SUBTYPES])
    print("MIMIC-IV not found — Part D skipped.")


✅ subtype markers: {'HE': 3415, 'variceal': 1451, 'ascites': 233}


In [18]:
# ---- Per-subtype early-warning models ----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

_co = pd.read_csv(COHORT_CSV, usecols=["patient_id","admittime","decompensated",
                                       "first_decomp_marker_time","los_days"])
_co["admittime"] = pd.to_datetime(_co["admittime"], errors="coerce")
_co["first_decomp_marker_time"] = pd.to_datetime(_co["first_decomp_marker_time"], errors="coerce")
base_feat = feat.merge(_co, on="patient_id", how="left", suffixes=("","_co"))
if len(submk):
    base_feat = base_feat.merge(submk, on="patient_id", how="left")

def subtype_eval(sub):
    tcol = f"{sub}_time"
    if tcol not in base_feat.columns:
        print(f"[{sub}] no marker column"); return None
    d = base_feat.copy()
    hrs = (d[tcol] - d["admittime"]).dt.total_seconds()/3600
    d["incident"] = (d[tcol].notna() & (hrs > LANDMARK_HOURS)).astype(int)
    prevalent = d[tcol].notna() & (hrs <= LANDMARK_HOURS)
    never_decomp = (d["decompensated"] == 0) & (d["first_decomp_marker_time"].isna())
    # cases = incident this subtype; controls = never decomp; drop prevalent + other-subtype-only
    keep = (d["incident"] == 1) | never_decomp
    keep &= ~prevalent
    dd = d[keep].copy()
    dd["y"] = dd["incident"].astype(int)
    dd = dd[(dd["y"] == 1) | (dd["los_days"]*24 > LANDMARK_HOURS)]   # controls must be at-risk
    nC, nK = int((dd.y==1).sum()), int((dd.y==0).sum())
    if nC < 30: print(f"[{sub}] too few incident cases ({nC}) — skipped"); return None
    y = dd["y"].values
    Xtr, Xte, ytr, yte = train_test_split(dd, y, test_size=TEST_FRAC, stratify=y, random_state=RANDOM_SEED)
    def predict(features):
        fs = [c for c in features if c in dd.columns]
        imp = SimpleImputer(strategy="median").fit(Xtr[fs]); sc = StandardScaler().fit(imp.transform(Xtr[fs]))
        clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(sc.transform(imp.transform(Xtr[fs])), ytr)
        return clf.predict_proba(sc.transform(imp.transform(Xte[fs])))[:,1]
    p_model = predict(MODEL_FEATURES)
    meld_col = "meld_max" if "meld_max" in dd.columns else "meld_last"
    p_meld = Xte[meld_col].values.astype(float)
    m = ~np.isnan(p_meld)                      # MELD-present test rows (paired comparison)
    def _ci(yt, s, n=2000, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed); a = []
        for _ in range(n):
            idx = rng.integers(0, len(yt), len(yt))
            if not (1 < yt[idx].sum() < len(idx)-1): continue
            a.append(roc_auc_score(yt[idx], s[idx]))
        return (float(np.median(a)), float(np.percentile(a,2.5)), float(np.percentile(a,97.5))) if a else (np.nan,)*3
    def _delta_ci(yt, s1, s2, n=2000, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed); a = []
        for _ in range(n):
            idx = rng.integers(0, len(yt), len(yt))
            if not (1 < yt[idx].sum() < len(idx)-1): continue
            a.append(roc_auc_score(yt[idx], s2[idx]) - roc_auc_score(yt[idx], s1[idx]))
        return (float(np.median(a)), float(np.percentile(a,2.5)), float(np.percentile(a,97.5))) if a else (np.nan,)*3
    a_full, lo_f, hi_f = _ci(yte, p_model)
    okm = m.sum() > 10 and len(np.unique(yte[m])) > 1
    a_meld, lo_m, hi_m = (_ci(yte[m], p_meld[m]) if okm else (np.nan,)*3)
    d_med, d_lo, d_hi  = (_delta_ci(yte[m], p_meld[m], p_model[m]) if okm else (np.nan,)*3)
    sig = "**" if (np.isfinite(d_lo) and (d_lo > 0 or d_hi < 0)) else "ns"
    print(f"[{sub:<9}] cases={nC:<5} controls={nK:<5}")
    print(f"           model AUC={a_full:.3f} [{lo_f:.3f}-{hi_f:.3f}] | MELD-Na AUC={a_meld:.3f} [{lo_m:.3f}-{hi_m:.3f}]")
    print(f"           model - MELD = {d_med:+.3f} [95% CI {d_lo:+.3f}, {d_hi:+.3f}]  {sig}")
    return dict(subtype=sub, n_cases=nC, n_controls=nK,
                model_auc=a_full, model_lo=lo_f, model_hi=hi_f,
                meld_auc=a_meld, meld_lo=lo_m, meld_hi=hi_m,
                delta=d_med, delta_lo=d_lo, delta_hi=d_hi, delta_sig=sig)

print("Composite reference: model AUC 0.726, MELD-Na 0.696\n")
results = [r for r in (subtype_eval(s) for s in SUBTYPES) if r]
if results:
    pd.DataFrame(results).to_csv(OUT/"subtype_results.csv", index=False)
    print("\n✅ Saved:", OUT/"subtype_results.csv")


Composite reference: model AUC 0.726, MELD-Na 0.696

[HE       ] cases=629   controls=2127 
           model AUC=0.764 [0.724-0.801] | MELD-Na AUC=0.745 [0.703-0.782]
           model - MELD = +0.019 [95% CI -0.010, +0.049]  ns
[variceal ] cases=176   controls=2127 
           model AUC=0.792 [0.727-0.847] | MELD-Na AUC=0.772 [0.707-0.830]
           model - MELD = +0.015 [95% CI -0.016, +0.048]  ns
[ascites  ] cases=43    controls=2127 
           model AUC=0.867 [0.782-0.930] | MELD-Na AUC=0.794 [0.637-0.932]
           model - MELD = +0.064 [95% CI -0.035, +0.196]  ns

✅ Saved: early_warning_output/subtype_results.csv


## Explainability — variance-normalized LLR vs SHAP
Compares the original variance-weighted LLR, a **unit-variance (LDA-style) normalized LLR**, 
and **SHAP** on the advanced-stage attribution, with Spearman rank agreement. Tests whether 
normalizing the LLR for feature scale reconciles it with SHAP.

In [19]:
# ============================================================
# EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP
# ============================================================
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    import shap as _shap; _HS = True
except Exception:
    _HS = False; print("shap not installed -> exact-linear fallback")

_P = stage_params; _S = _seqs; _OBS = list(OBS_STAGE); _K = K_STAGE
_mu, _var = _P["mu"], _P["var"]

def _llr_feat(o, tgt, ref=0, unit=False):
    out = {}
    for j, f in enumerate(_OBS):
        if np.isnan(o[j]): continue
        if unit:   # unit-variance / LDA-style: scale-invariant on standardized obs
            out[f] = (_mu[tgt,j]-_mu[ref,j]) * (o[j]-0.5*(_mu[tgt,j]+_mu[ref,j]))
        else:      # original variance-weighted LLR
            lt = -0.5*(np.log(2*np.pi*_var[tgt,j]) + (o[j]-_mu[tgt,j])**2/_var[tgt,j])
            lr = -0.5*(np.log(2*np.pi*_var[ref,j]) + (o[j]-_mu[ref,j])**2/_var[ref,j])
            out[f] = lt - lr
    return out

_raw = {f: [] for f in _OBS}; _nrm = {f: [] for f in _OBS}; _X = []; _Y = []
for s in _S:
    g, st, _ = posterior_stage(_P, s["O"]); hard = np.argmax(g, axis=1)
    for t in range(len(st)):
        _X.append(s["O"][t]); _Y.append(int(hard[t] == _K-1))
        if hard[t] == _K-1:
            for f, v in _llr_feat(s["O"][t], _K-1, 0, False).items(): _raw[f].append(abs(v))
            for f, v in _llr_feat(s["O"][t], _K-1, 0, True ).items(): _nrm[f].append(abs(v))
Xv = pd.DataFrame(_X, columns=_OBS); Xv = Xv.fillna(Xv.median()); Yv = np.array(_Y)
print(f"visit-level: {len(Xv):,} visits | advanced (top stage) = {int(Yv.sum()):,} ({Yv.mean()*100:.1f}%)")

llr_raw  = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _raw.items()})
llr_norm = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _nrm.items()})
_gbm = GradientBoostingClassifier(max_depth=3, n_estimators=200, random_state=0).fit(Xv, Yv)
if _HS:
    _r = _shap.TreeExplainer(_gbm).shap_values(Xv)
    if isinstance(_r, list): _r = _r[-1]
    _r = np.asarray(_r)
    if _r.ndim == 3: _r = _r[:, :, -1]
    shap_imp = pd.Series(np.abs(_r).mean(0), index=_OBS)
else:
    _lin = LogisticRegression(max_iter=2000).fit(Xv, Yv)
    shap_imp = pd.Series(np.abs(_lin.coef_[0])*Xv.std().values, index=_OBS)

cmp = pd.DataFrame({"LLR_raw":llr_raw, "LLR_norm":llr_norm, "SHAP":shap_imp})
for c in ["LLR_raw","LLR_norm","SHAP"]: cmp[c+"_rk"] = cmp[c].rank(ascending=False)
cmp = cmp.sort_values("SHAP", ascending=False)
rho_raw  = spearmanr(cmp["LLR_raw_rk"],  cmp["SHAP_rk"]).correlation
rho_norm = spearmanr(cmp["LLR_norm_rk"], cmp["SHAP_rk"]).correlation
print("\n=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===")
print(cmp[["LLR_raw","LLR_norm","SHAP","LLR_raw_rk","LLR_norm_rk","SHAP_rk"]].round(3).to_string())
print(f"\nSpearman  SHAP vs raw-LLR  = {rho_raw:+.3f}")
print(f"Spearman  SHAP vs norm-LLR = {rho_norm:+.3f}")
print("  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a variance artifact.")


visit-level: 75,250 visits | advanced (top stage) = 23,356 (31.0%)

=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===
            LLR_raw  LLR_norm   SHAP  LLR_raw_rk  LLR_norm_rk  SHAP_rk
Creatinine    7.890     4.127  2.407         3.0          2.0      1.0
Bilirubin   463.062     7.148  2.060         1.0          1.0      2.0
INR          18.963     2.224  0.860         2.0          3.0      3.0
Lactate       2.972     1.410  0.477         5.0          4.0      4.0
Albumin       0.121     0.130  0.129         8.0          8.0      5.0
MAP           0.531     0.597  0.125         6.0          6.0      6.0
Ammonia       4.363     1.352  0.065         4.0          5.0      7.0
Sodium        0.410     0.367  0.032         7.0          7.0      8.0

Spearman  SHAP vs raw-LLR  = +0.690
Spearman  SHAP vs norm-LLR = +0.810
  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a variance artifact.
